In [ ]:
import pandas as pd
import numpy as np
import gc

# -----------------------------
# 1. Load and Clean Data
# -----------------------------

def load_data(num_parts=5):
    
    
    
    """Load and concatenate cleaned data from Parquet files."""
    path = "../data/checkpoints/checkpoints/01_cleaned_data/01_cleaned_data_part_{}.parquet"
    dfs = [pd.read_parquet(path.format(i)) for i in range(1, num_parts + 1)]
    return pd.concat(dfs, ignore_index=True)


def clean_references(refs):
    """Ensure references are a clean list of non-empty strings."""
    if isinstance(refs, (list, np.ndarray)):
        return [str(r) for r in refs if r and str(r).strip()]
    return []


# Load and perform initial cleaning
df = load_data()
df = df.drop(columns=["issn", "isbn"], errors="ignore")
df = df.dropna(subset=["id", "references"]).copy()
df["references"] = df["references"].apply(clean_references)
df = df[df["references"].map(len) > 0].reset_index(drop=True)

print(f"Loaded {len(df)} articles after initial cleaning.")
gc.collect()

# -----------------------------
# 2. Create Positive and Negative Citation Pairs
# -----------------------------

article_features = df.drop(columns=["references"]).add_suffix("_article")
article_features = article_features.rename(columns={"id_article": "article_id"})

ref_features = df.drop(columns=["references"]).add_suffix("_ref")
ref_features = ref_features.rename(columns={"id_ref": "ref_id"})

# Positive pairs (article_id, true ref_id)
positive_pairs = df[["id", "references"]].explode("references")
positive_pairs = positive_pairs.rename(columns={"id": "article_id", "references": "ref_id"})
positive_pairs = positive_pairs.dropna(subset=["ref_id"])

valid_ref_ids = set(df["id"])
positive_pairs = positive_pairs[positive_pairs["ref_id"].isin(valid_ref_ids)].reset_index(drop=True)

# Negative pairs (article_id, sampled fake ref_id)
all_ids = df["id"].to_numpy()
true_refs_map = df.set_index("id")["references"].apply(set).to_dict()

fake_ref_ids = []
for article_id in positive_pairs["article_id"].to_numpy():
    true_refs = true_refs_map.get(article_id, set())
    fake_id = np.random.choice(all_ids)
    while fake_id == article_id or fake_id in true_refs:
        fake_id = np.random.choice(all_ids)
    fake_ref_ids.append(fake_id)

negative_pairs = positive_pairs[["article_id"]].copy()
negative_pairs["ref_id"] = fake_ref_ids

# -----------------------------
# 3. Assemble Final Training Data
# -----------------------------

df_pos = (
    positive_pairs.merge(article_features, on="article_id", how="left")
    .merge(ref_features, on="ref_id", how="inner")
)
df_pos["is_reference_valid"] = 1

df_neg = (
    negative_pairs.merge(article_features, on="article_id", how="left")
    .merge(ref_features, on="ref_id", how="inner")
)
df_neg["is_reference_valid"] = 0

df_training = pd.concat([df_pos, df_neg], ignore_index=True)
rows_before_filters = len(df_training)

# Keep only rows with mandatory identifiers/label
mandatory_cols = ["article_id", "ref_id", "is_reference_valid"]
existing_mandatory_cols = [c for c in mandatory_cols if c in df_training.columns]
df_training = df_training.dropna(subset=existing_mandatory_cols)
rows_after_mandatory = len(df_training)

# Apply year consistency only when both years are present
if "year_article" in df_training.columns and "year_ref" in df_training.columns:
    year_article = pd.to_numeric(df_training["year_article"], errors="coerce")
    year_ref = pd.to_numeric(df_training["year_ref"], errors="coerce")

    valid_year_mask = year_article.isna() | year_ref.isna() | (year_article >= year_ref)
    df_training = df_training[valid_year_mask]

# Stable ordering for reproducibility
if "year_article" in df_training.columns:
    df_training = df_training.sort_values("year_article", na_position="last")

df_training = df_training.reset_index(drop=True)

print(f"Positive samples: {len(df_pos)}")
print(f"Negative samples: {len(df_neg)}")
print(f"Rows before final filters: {rows_before_filters}")
print(f"Rows after mandatory-field filter: {rows_after_mandatory}")
print(f"Final training dataset size: {len(df_training)}")

# Clean up memory
del df, df_pos, df_neg, positive_pairs, negative_pairs, article_features, ref_features
gc.collect()


Loaded 421479 articles after initial cleaning.
Positive samples: 391846
Negative samples: 391846
Rows before final filters: 783692
Rows after mandatory-field filter: 783692
Final training dataset size: 632779


0

In [6]:
# -----------------------------
# 4. Feature Engineering
# -----------------------------

# Separate features (X) from the target variable (y)
X = df_training.drop(columns=['is_reference_valid', 'article_id', 'ref_id']).copy()
y = df_training['is_reference_valid'].copy()

def extract_author_names(authors):
    """Extracts author names from a dictionary or list of dictionaries."""
    if isinstance(authors, dict):
        return authors.get('name', '')
    if isinstance(authors, list):
        return ', '.join(a.get('name', '') for a in authors if 'name' in a)
    return ''

def combine_text_features(df, suffix):
    """Combines title, abstract, keywords, and authors into a single text string."""
    title = df.get(f'title{suffix}', '')
    abstract = df.get(f'abstract{suffix}', '')
    keywords = df.get(f'keywords{suffix}', '')
    authors = extract_author_names(df.get(f'authors{suffix}', ''))
    
    return f"{title} {abstract} {keywords} {authors}"

# Create combined text features for articles and references
X['vector_text_article'] = X.apply(lambda row: combine_text_features(row, '_article'), axis=1)
X['vector_text_ref'] = X.apply(lambda row: combine_text_features(row, '_ref'), axis=1)

# Fill any potential NaN values
X['vector_text_article'] = X['vector_text_article'].fillna('').astype(str)
X['vector_text_ref'] = X['vector_text_ref'].fillna('').astype(str)

# Clean up memory
del df_training
gc.collect()


0

In [7]:
import scipy.sparse as sp
import importlib
import feature_extractor
import gc

# Reload local module to pick up latest changes during notebook runs
importlib.reload(feature_extractor)
FeatureExtractor = feature_extractor.FeatureExtractor

# -----------------------------
# 5. Vectorize Text and Combine Features
# -----------------------------

# Initialize the feature extractor with a limit on the number of features
feature_extractor = FeatureExtractor(max_features=256)

# Get the text data for articles and references
articles_text = X['vector_text_article'].tolist()
refs_text = X['vector_text_ref'].tolist()

# Fit the vectorizer and transform the text data
feature_extractor.fit(articles_text, refs_text)
X_articles, X_refs, sims = feature_extractor.transform(articles_text, refs_text)

# Define columns to drop to keep only numeric metadata
non_numeric_cols = [
    'title_article', 'abstract_article', 'keywords_article', 'lang_article',
    'authors_article', 'venue_article', 'doc_type_article', 'doi_article', 'url_article',
    'title_ref', 'abstract_ref', 'keywords_ref', 'lang_ref',
    'authors_ref', 'venue_ref', 'doc_type_ref', 'doi_ref', 'url_ref',
    'vector_text_article', 'vector_text_ref'
]

# Isolate metadata, ensuring it's all numeric
meta = X.drop(columns=non_numeric_cols, errors='ignore').copy()
for col in meta.columns:
    meta[col] = pd.to_numeric(meta[col], errors='coerce').fillna(0.0)

# Convert metadata to a sparse matrix
if meta.shape[1] > 0:
    X_meta = sp.csr_matrix(meta.to_numpy(dtype=np.float32))
else:
    X_meta = sp.csr_matrix((len(X), 0), dtype=np.float32)

# Reshape similarities to be a column vector
sims_col = sp.csr_matrix(np.asarray(sims, dtype=np.float32).reshape(-1, 1))

# Combine all features into a single sparse matrix
X_model = sp.hstack([X_meta, X_articles.tocsr(), X_refs.tocsr(), sims_col], format='csr')
y_model = y.to_numpy()

print(f"Final feature matrix shape: {X_model.shape}")

# Clean up memory
del X, y, articles_text, refs_text, X_articles, X_refs, sims, meta, X_meta, sims_col
gc.collect()

Final feature matrix shape: (632779, 525)


0

In [8]:
from sklearn.model_selection import train_test_split

# -----------------------------
# 6. Split Data for Training and Testing
# -----------------------------

X_train, X_test, y_train, y_test = train_test_split(
    X_model, y_model, 
    test_size=0.2, 
    random_state=42, 
    stratify=y_model  # Ensure proportional class representation
)

print(f"Train set shape: {X_train.shape}")
print(f"Test set shape: {X_test.shape}")

# Since X_model is no longer needed, free up memory
del X_model, y_model
gc.collect()


Train set shape: (506223, 525)
Test set shape: (126556, 525)


0

In [ ]:
import xgboost as xgb
from sklearn.metrics import classification_report
import time
import gc

# -----------------------------
# 7. Train and Evaluate XGBoost Classifier
# -----------------------1
# Keep matrices sparse to avoid memory blow-ups
X_train_sparse = X_train.astype('float32')
X_test_sparse = X_test.astype('float32')

# Free up memory from original matrices
del X_train, X_test
gc.collect()

# Configure the XGBoost classifier for performance
# 'hist' is a fast and efficient tree construction method
clf = xgb.XGBClassifier(
    tree_method='hist',      # Use histogram-based algorithm for speed
    predictor='auto',        # Automatically select the best predictor
    random_state=42,
    n_estimators=100,
    learning_rate=0.1,
    max_depth=6,
    n_jobs=-1,               # Use all available CPU cores
    subsample=0.8,
    colsample_bytree=0.8
)

print("Training XGBoost with sparse matrices...")
start_time = time.time()

clf.fit(X_train_sparse, y_train, verbose=False)

elapsed_time = time.time() - start_time
print(f"✓ Training completed in {elapsed_time:.2f}s")

# Generate predictions on the test set
print("Generating predictions...")
y_pred = clf.predict(X_test_sparse)

# Display classification metrics
print("\n" + "="*60)
print("RESULTS - XGBoost (tree_method='hist', n_jobs=-1)")
print("="*60)
print(classification_report(y_test, y_pred, digits=4))
print("="*60)

Training XGBoost with sparse matrices...


/home/tommaso/miniconda3/envs/hack_03/lib/python3.11/site-packages/xgboost/training.py:200: UserWarning: [22:17:11] WARNING: /home/task_177203090464722/croot/xgboost-split_1772031726713/work/src/learner.cc:782: 
Parameters: { "predictor" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


✓ Training completed in 13.92s
Generating predictions...

RESULTS - XGBoost (tree_method='hist', n_jobs=-1)
              precision    recall  f1-score   support

           0     0.8673    0.8790    0.8731     48943
           1     0.9230    0.9152    0.9191     77613

    accuracy                         0.9012    126556
   macro avg     0.8952    0.8971    0.8961    126556
weighted avg     0.9015    0.9012    0.9013    126556

